# Track A on Colab GPU — NLI scoring + first submission

Runs the Track A pipeline from the repo on a GPU runtime:
1. mounts Drive (data lives in `Bengali-LLM-Hallucination-Detection/data/` there)
2. clones/pulls the GitHub repo
3. scores train + test rows with the NLI model (minutes on a T4, ~1h on CPU)
4. builds + validates `submissions/nli_ctx_majority_cb.csv`
5. copies the score caches and the submission back to Drive

**Runtime → Change runtime type → T4 GPU** before running.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Bengali-LLM-Hallucination-Detection'
import os
assert os.path.exists(f'{DRIVE_DIR}/data/test set.csv'), 'competition data not found on Drive'

Mounted at /content/drive


In [2]:
%cd /content
!git clone https://github.com/FaiyazAwsaf/Bengali-LLM-Hallucination-Detection.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo

# data/ is gitignored, so bring it in from Drive
!mkdir -p data submissions
!cp "{DRIVE_DIR}/data/dataset samples.json" "{DRIVE_DIR}/data/test set.csv" "{DRIVE_DIR}/data/sample submission.csv" data/

# reuse previously computed score caches from Drive if they exist
!mkdir -p data/cache
!cp "{DRIVE_DIR}/cache/"*.csv data/cache/ 2>/dev/null || echo 'no cache on Drive yet'

!pip install -q -U transformers sentencepiece

/content
/content/repo
no cache on Drive yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 103.4 MB/s eta 0:00:00


In [3]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable a GPU runtime')

GPU: Tesla T4


In [4]:
# score the 299 labeled rows + run repeated CV (fast; verifies everything works)
!cd src && python track_a_nli.py cv

config.json: 100% 1.09k/1.09k [00:00<00:00, 2.23MB/s]
tokenizer_config.json: 100% 467/467 [00:00<00:00, 2.31MB/s]
spm.model: 100% 4.31M/4.31M [00:01<00:00, 4.11MB/s]
tokenizer.json: 100% 16.3M/16.3M [00:00<00:00, 21.2MB/s]
added_tokens.json: 100% 23.0/23.0 [00:00<00:00, 157kB/s]
special_tokens_map.json: 100% 173/173 [00:00<00:00, 1.18MB/s]
model.safetensors: 100% 558M/558M [00:03<00:00, 185MB/s]
Loading weights: 100% 202/202 [00:00<00:00, 5697.94it/s]
  scored 32/130 (2s)
  scored 64/130 (2s)
  scored 96/130 (3s)
  scored 128/130 (3s)
  scored 130/130 (4s)

=== nli_zeroshot_ctx+majority_cb ===
  overall      Macro-F1: 0.595 +/- 0.029
  context      Macro-F1: 0.657 +/- 0.085
  closed_book  Macro-F1: 0.345 +/- 0.003
  context confusion (rows=true 0/1, cols=pred 0/1):
[[113 122]
 [ 70 345]]
  closed_book confusion (rows=true 0/1, cols=pred 0/1):
[[445   0]
 [400   0]]

full-train calibrated threshold: 0.000


In [5]:
# score the 1,361 test context rows (the expensive step — minutes on T4)
!cd src && python track_a_nli.py test | tail -3

Loading weights: 100% 202/202 [00:00<00:00, 5314.77it/s]
  scored 1344/1361 (19s)
  scored 1361/1361 (19s)
saved /content/repo/data/cache/nli_test_scores.csv


In [6]:
# build + validate the submission CSV
!cd src && python combine_and_predict.py nli_ctx_majority_cb.csv

threshold (fit on all 130 labeled context rows): 0.000
OK: /content/repo/submissions/nli_ctx_majority_cb.csv (2516 rows, mean label 0.353, context mean 0.652)


In [7]:
# persist results to Drive: score caches (reusable) + the submission file
!mkdir -p "{DRIVE_DIR}/cache" "{DRIVE_DIR}/submissions"
!cp data/cache/nli_*.csv "{DRIVE_DIR}/cache/"
!cp submissions/nli_ctx_majority_cb.csv "{DRIVE_DIR}/submissions/"
!ls -la "{DRIVE_DIR}/cache" "{DRIVE_DIR}/submissions"

/content/drive/MyDrive/Bengali-LLM-Hallucination-Detection/cache:
total 60
-rw------- 1 root root 54839 Jul  8 13:08 nli_test_scores.csv
-rw------- 1 root root  5499 Jul  8 13:08 nli_train_scores.csv

/content/drive/MyDrive/Bengali-LLM-Hallucination-Detection/submissions:
total 17
-rw------- 1 root root 16514 Jul  8 13:08 nli_ctx_majority_cb.csv


Done. Back on the local machine, download the caches + submission from Drive into `data/cache/`
and `submissions/`, then submit with:

```
kaggle competitions submit -c <competition> -f submissions/nli_ctx_majority_cb.csv -m "Track A NLI + majority closed-book"
```